<a href="https://colab.research.google.com/github/BillPapakyriakou/DataMining-Notebooks/blob/main/assignment_1B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Πρώτη Σειρά Ασκήσεων – Μέρος B
---
### 5324 - Παπακυριακού Βασίλειος

### 5361 - Άννα Ταρασίδου



---
##**Βήμα 1 - Φόρτωση και καθαρισμός των δεδομένων**

In [7]:
import pandas as pd  # pandas
import numpy as np   # numpy
import ast           # ast : abstract syntax trees : χρήση συνάρτησης που μετατρέπει literal σε object, πχ list

dataframe = pd.read_csv("movies_metadata.csv", on_bad_lines="skip")  # φορτώνει το αρχείο με τα metadata και αγνοεί τις κακές γραμμές

columns_used = ["id", "title", "vote_count", "runtime", "vote_average",
    "revenue", "budget", "release_date", "genres"]    # λίστα με τα πεδία που θα χρησιμοποιήσουμε στο ερωτήματα 1-7

dataframe = dataframe[columns_used].copy()

numeric_columns = ["vote_count", "runtime", "vote_average", "revenue", "budget"]  # πεδία που πρέπει να είναι αριθμοί

for col in numeric_columns:   # για κάθε στήλη numeric
    dataframe[col] = pd.to_numeric(dataframe[col], errors="coerce")   # μετέτρεψε σε numeric (αν δεν είναι ήδη)
                                                                      # όπου δεν γίνεται - μετατροπή σε NaN

dataframe["release_date"] = pd.to_datetime(dataframe["release_date"], errors="coerce")  # μετατροπή σε data format (αν δεν είναι ήδη)
                                                                                        # όπου δεν γίνεται - μετατροπή σε NaN

# κάνει το πεδίο genres αντικείμενο python (λίστα από dictionaries αντί για string)
def fix_genres(val):
    if pd.isna(val):
        return np.nan
    try:
        # μετατροπή string τύπου [{"id":..., "name":...}] list από dictionaries
        return ast.literal_eval(val)
    except:
        return np.nan  # αν δεν γίνεται τότε NaN

dataframe["genres"] = dataframe["genres"].apply(fix_genres)

# αφαιρεί γραμμές με NaN (not a number) από τις columns που μας ενδιαφέρουν
dataframe.dropna(subset=["id", "title", "vote_count", "runtime", "vote_average",
    "revenue", "budget", "release_date", "genres"], inplace=True)

# αφαίρεση γραμμών με παράλογες τιμές

# runtime > 0 και < 300
dataframe = dataframe[(dataframe["runtime"] > 0) & (dataframe["runtime"] < 300)]

# vote_average μεταξύ 0 και 10
dataframe = dataframe[dataframe["vote_average"].between(0, 10)]

# budget και revenue μη αρνητικά
dataframe = dataframe[(dataframe["budget"] >= 0) & (dataframe["revenue"] >= 0)]

# τυπώνουμε συνολικό αριθμό rows και cols των δεδομένων
print("Τελικός αριθμός γραμμών:", dataframe.shape[0])
print("Τελικός αριθμός στηλών:", dataframe.shape[1])

# μερικά extra checks για το dataframe - TODO - περαιτέρω ανάλυση
#dataframe.info()
#dataframe.isna().sum()
#dataframe[numeric_columns].describe()
#dataframe["genres"].head(10)



/tmp/ipython-input-634557871.py:5: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  dataframe = pd.read_csv("movies_metadata.csv", on_bad_lines="skip")  # φορτώνει το αρχείο με τα metadata και αγνοεί τις κακές γραμμές


Τελικός αριθμός γραμμών: 43476
Τελικός αριθμός στηλών: 9


##**Βήμα 2 - Ανάλυση κατανομής του πεδίου vote_counts**